In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
db_path = Path(r"D:\BDDPlabacomCoordinador") 

In [17]:
lista_cmg = []
dates = []
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue

    origin, date, version = folder_date.name.split("_")
    dates.append(int(date))
    print(f"Mirando la fecha  {date}")
    
    target_dir = Path(r"D:\BDDPlabacomCoordinador") / f"PLABACOM_{date}_BD01" / "01 Cmg"
    
    csv_files = list(target_dir.glob("*.csv"))
    
    if not csv_files:
        print("csv not found")
        continue
        
    df_cmg = pd.read_csv(csv_files[0], sep=";")

    df_cmg = df_cmg.groupby(by=["nombre_barra", "tension", "nombre_barra_cmg", "HORA"], as_index = False).agg({"FECHA":"last", 
                                                                                         "CMg[CLP/KWh]":"mean", 
                                                                                         "CMg[USD/MWh]":"mean"}, as_index=False)
    df_cmg["FECHA"] = df_cmg["FECHA"].astype(str)
    df_cmg[["Año", "Mes"]] = df_cmg["FECHA"].str.extract(r"(\d{4})(\d{2})\d{2}")
    df_cmg = df_cmg.drop(columns = ["FECHA"])
    lista_cmg.append(df_cmg)

df_cmg = pd.concat(lista_cmg, ignore_index=True)
df_cmg["Periodo"] = f"{min(dates)}_{max(dates)}"



Mirando la fecha  2505
Mirando la fecha  2506
Mirando la fecha  2507
Mirando la fecha  2508
Mirando la fecha  2509
Mirando la fecha  2510
Mirando la fecha  2511
Mirando la fecha  2512
Mirando la fecha  2601
Mirando la fecha  2602
Mirando la fecha  2603
Mirando la fecha  2604


In [18]:
df_cmg.head()

,nombre_barra,tension,nombre_barra_cmg,HORA,CMg[CLP/KWh],CMg[USD/MWh],Año,Mes,Periodo
0,A,100,A_____________100,1,71.568910,75.925060,2025,05,2505_2604
1,A,100,A_____________100,2,69.478102,73.703558,2025,05,2505_2604
2,A,100,A_____________100,3,77.806368,82.567770,2025,05,2505_2604
3,A,100,A_____________100,4,79.085300,83.945624,2025,05,2505_2604
4,A,100,A_____________100,5,72.378804,76.801453,2025,05,2505_2604


In [19]:
new_cols_mes = ["CMg[CLP/KWh]_mes", "CMg[USD/MWh]_mes"]
cols_cmg = ["CMg[CLP/KWh]", "CMg[USD/MWh]"]
grupos_mes = ["nombre_barra", "tension", "nombre_barra_cmg", "Mes"]

new_cols_period = ["CMg[CLP/KWh]_period", "CMg[USD/MWh]_period"]
cols_cmg = ["CMg[CLP/KWh]", "CMg[USD/MWh]"]
grupos_period = ["nombre_barra", "tension", "nombre_barra_cmg", "Periodo"]

df_cmg[new_cols_mes] = df_cmg.groupby(grupos_mes)[cols_cmg].transform("mean")
df_cmg[new_cols_period] = df_cmg.groupby(grupos_period)[cols_cmg].transform("mean")

In [21]:
df_cmg.head()

,nombre_barra,tension,nombre_barra_cmg,HORA,CMg[CLP/KWh],CMg[USD/MWh],Año,Mes,Periodo,CMg[CLP/KWh]_mes,CMg[USD/MWh]_mes,CMg[CLP/KWh]_period,CMg[USD/MWh]_period
0,A,100,A_____________100,1,71.568910,75.925060,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
1,A,100,A_____________100,2,69.478102,73.703558,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
2,A,100,A_____________100,3,77.806368,82.567770,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
3,A,100,A_____________100,4,79.085300,83.945624,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
4,A,100,A_____________100,5,72.378804,76.801453,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068


In [22]:
df_cmg.to_parquet(Path(r"D:\ProyectoAnalisisElectrico\Cmg\Cmg_period.parquet"), engine="pyarrow")